<a href="https://colab.research.google.com/github/TWalkerSMCM/icsa-scraper/blob/main/examples/quickstart.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# icsa-scraper · Quickstart

A guided tour of the analysis layer: scrape a season from [scores.collegesailing.org](https://scores.collegesailing.org) into a queryable `Dataset`, explore it with pandas, then peek under the hood at the layers `Dataset` is built on (`Client`, `urls`, `fleet_scores`, and the pure parsers).

`scraper.load()` does all the fetching + assembly for you — no manual page-by-page stitching required.

## Install

The `[fetch]` extra adds `httpx` + `tenacity` — `scraper.load()` needs those to fetch pages itself.

In [ ]:
# Fresh runtimes (Colab) need the library. Guard on the distribution name so
# re-running locally can't clobber an editable dev install; to force-update a
# cached Colab runtime, restart it (or run the %pip line with --force-reinstall).
from importlib.metadata import PackageNotFoundError, version

try:
    version("icsa-scraper")
except PackageNotFoundError:
    %pip install -q "icsa-scraper[fetch] @ git+https://github.com/TWalkerSMCM/icsa-scraper"

## Optional: persist the cache on Colab

`scraper.load()` caches every fetched page under `./.scraper_cache`, so re-running a cell (or the whole notebook) after the first pass is instant. Colab wipes that folder on runtime reset — mount Drive and point the cache there first if you want it to survive. Set `SCRAPER_CACHE_DIR` **before** importing `scraper`:

```python
# from google.colab import drive
# drive.mount('/content/drive')
# import os
# os.environ['SCRAPER_CACHE_DIR'] = '/content/drive/MyDrive/icsa_cache'
```

## Load a season into a `Dataset`

`scraper.load(season)` scrapes the season index, fetches every regatta's pages, and assembles them all into one in-memory `Dataset`. That's a lot of requests for a full season, so here we scope it to a few regattas with `only=[...]` — drop that argument to load everything. `workers` controls how many regattas are fetched concurrently (default 16; the work is latency-bound against a static site, so this is safe), and `progress=True` shows a per-regatta progress bar (tqdm when installed — Colab has it — else a plain counter).

In [ ]:
import scraper

SEASON = "s26"  # f25 = Fall 2025, s26 = Spring 2026
REGATTAS = [
    "open-fleet-race-national-championships",
    "open-east-national-semis",
    "open-west-national-semis",
]

data = scraper.load(SEASON, only=REGATTAS, workers=16, progress=True)
data

## Results — one row per (regatta, school)

`results_frame()` is the pandas escape hatch out of `data.results`. `start_time` comes back as a real `datetime64` column, ready to sort/group on. `total` is `NaN` for team racing (there's no fleet points total to report).

In [ ]:
results = data.fleet().results_frame()
results.sort_values(["regatta_slug", "place"]).head(10)

## Regattas at a glance

`regattas_frame()` gives one row per regatta — identity, scoring type, dates, `team_count`, and (when the overview page carried it) `boat`/`participant`/`regatta_type`.

In [ ]:
data.regattas_frame()

## Discovery and filtering

`data.schools` / `data.sailors` list the slugs present in the current (possibly narrowed) dataset. `.fleet()`/`.team()` and `.school(slug)`/`.sailor(slug)` chain to narrow every projection (`regattas`, `results`, `finishes`, `sailor_races`) together.

In [ ]:
print(len(data.schools), "schools,", len(data.sailors), "sailors")

navy = data.school("navy")
navy.results_frame()[["regatta_name", "place", "total"]]

## Per-sailor results

`sailor_races_frame()` is the RP↔finish join: which sailor sailed which race, and the place their boat earned — pre-joined with the regatta's `regatta_type`/`boat`/`participant`/`start_time`. This is the row type behind skipper ELO and head-to-head.

In [ ]:
sailor_races = data.sailor_races_frame()
sailor_races.sort_values("start_time").head(10)

## Quick chart

In [ ]:
import matplotlib.pyplot as plt

top = (
    results.dropna(subset=["total"])
    .sort_values("total")
    .drop_duplicates("school_slug")
    .head(10)
    .iloc[::-1]
)
plt.figure(figsize=(8, 4))
plt.barh(top["school"], top["total"])
plt.xlabel("Total points (lower = better)")
plt.title(f"{SEASON} - best single-regatta finishes (sample)")
plt.tight_layout()
plt.show()

## Head-to-head, without scraping a season

If you only want to compare two sailors, `scraper.head_to_head()` fetches just their two profile pages and intersects the regattas they shared — no season load needed.

In [ ]:
# Pick two skippers who sailed the same regatta-division, so they overlap.
from collections import defaultdict

by_div = defaultdict(set)
for r in data.sailor_races:
    if r.boat_role == "skipper":
        by_div[(r.regatta_slug, r.division)].add(r.sailor_slug)
a, b = sorted(max(by_div.values(), key=len))[:2]

h = scraper.head_to_head(a, b)
print(
    a, "vs", b, "-", len(h.shared), "shared regatta-divisions;", "ahead:", h.a_ahead, "-", h.b_ahead
)
h.shared_frame()

## Under the hood: the layers `load()` is built on

`Dataset` is just glue over three lower layers you can call directly:

1. **`scraper.urls`** — pure functions for site-relative paths.
2. **`scraper.Client`** — a cached, rate-limited `httpx` session (`client.fetch(path)`).
3. **`scraper.fleet_scores`** (or `team_scores`) — HTML in, `RegattaScores` model out.

This is the same path `load()` takes per regatta, just for one at a time.

In [ ]:
with scraper.Client() as client:
    slug = REGATTAS[0]
    fs_html = client.fetch(scraper.urls.full_scores(SEASON, slug))
    reg = scraper.fleet_scores(fs_html, season=SEASON, slug=slug)

print("Regatta:     ", reg.name)
print("Scoring:     ", reg.scoring_type)
print("Teams:       ", len(reg.teams))
print("Winner:      ", reg.teams[0].school, reg.teams[0].total)

### Even lower: a single raw parser call

`scraper.parsers.*` are pure functions over HTML — no network, no assembly, just one page's structure. `fleet_scores` above calls `full_scores.parse` internally; here it is directly.

In [ ]:
from scraper.parsers import full_scores

div_scores = full_scores.parse(fs_html)
print(len(div_scores), "team-division score rows")
div_scores[0]

---
**Recap:** `scraper.load(season)` → a `Dataset` you can filter (`.fleet()`, `.school()`, `.sailor()`) and export to pandas (`results_frame()`, `sailor_races_frame()`, `regattas_frame()`). `scraper.head_to_head()` skips the season load entirely for a two-sailor comparison. Underneath both: `scraper.Client` + `scraper.urls` to fetch, `fleet_scores`/`team_scores` to assemble, and the pure `scraper.parsers.*` doing the actual HTML reading. Swap `SEASON`/`REGATTAS` to explore anything else on the site.